<img src='https://drive.google.com/uc?export=view&id=1Hh_G3M13P9xSNgSiQ-WnALg93XwK_hG8' alt='Encabezado MLDS' width='100%'></img>

# **Despliegue del Modelo**
---

Este notebook documenta el despliegue del modelo de verificación de firmas adulteradas como una aplicación web interactiva usando **Streamlit**.

> **Prerrequisito:** Haber ejecutado `scripts/4_modeling.ipynb` completo. El archivo `modelo_siames_firmas_final.keras` debe existir en el directorio de trabajo de Colab antes de continuar con la celda 1.3.

## **0. Integrantes del equipo de trabajo**
---

1. Ivonne Alexandra Guevara Prieto - CC 1032376081

## **1. Setup**
---

### **1.1. Instalación de dependencias**
---

In [ ]:
# Instalar dependencias
!pip install -q streamlit kagglehub opencv-python-headless
print('✅ Dependencias instaladas.')

### **1.2. Librerías y semillas**
---

In [ ]:
import tensorflow as tf
import keras
import numpy as np
import matplotlib.pyplot as plt
import os, random
import cv2
import pandas as pd
%matplotlib inline
plt.style.use('ggplot')

# Configuración de semillas para reproducibilidad
random.seed(0)
keras.utils.set_random_seed(0)
tf.random.set_seed(0)
np.random.seed(0)

import warnings
warnings.filterwarnings('ignore')

print('TensorFlow:', tf.__version__)
print('Keras     :', keras.__version__)

### **1.3. Carga del modelo entrenado**
---

El modelo debe haber sido guardado en `scripts/4_modeling.ipynb` con:
```python
mejor_modelo_m1.save('modelo_siames_firmas_final.keras')
```
Si el archivo no existe en Colab, súbelo manualmente con la celda siguiente.

In [ ]:
MODEL_PATH = 'modelo_siames_firmas_final.keras'

# Opción A: el modelo ya existe en el directorio de Colab
if os.path.exists(MODEL_PATH):
    print(f'✅ Modelo encontrado en: {MODEL_PATH}')
else:
    # Opción B: subir el modelo manualmente desde tu equipo
    print(f'⚠️  Modelo no encontrado. Ejecuta la celda siguiente para subirlo.')

In [ ]:
# Ejecutar SOLO si el modelo no está en Colab todavía
# Sube el archivo 'modelo_siames_firmas_final.keras' desde tu equipo
if not os.path.exists(MODEL_PATH):
    from google.colab import files
    print('Por favor sube el archivo modelo_siames_firmas_final.keras:')
    uploaded = files.upload()
    print('✅ Archivo subido:', list(uploaded.keys()))

In [ ]:
# Cargar el modelo
modelo = keras.models.load_model(MODEL_PATH)
modelo.summary()
print(f'\n✅ Modelo cargado correctamente desde: {MODEL_PATH}')

## **2. Funciones de preprocesamiento e inferencia**
---

Estas funciones están también disponibles como módulos Python en `src/firmas_adulteradas/preprocessing.py` y `src/firmas_adulteradas/predict.py`.

In [ ]:
def preprocesar_imagen(ruta_img, tam=(105, 105)):
    """
    Preprocesa una imagen de firma para la Red Siamesa:
    1. Lectura en escala de grises.
    2. Redimensionamiento a 105x105 px.
    3. Normalización al rango [0.0, 1.0].
    4. Reshape a (105, 105, 1).
    """
    img = cv2.imread(ruta_img, cv2.IMREAD_GRAYSCALE)
    img = cv2.resize(img, tam)
    img = img / 255.0
    return img.reshape(tam[0], tam[1], 1).astype(np.float32)


# Umbrales de decisión (calibrables según política del negocio)
UMBRAL_GENUINO     = 0.40
UMBRAL_FALSIFICADA = 0.60


def verificar_firma(ruta_referencia, ruta_sospechosa, modelo):
    """
    Verifica si un par de firmas es genuino o falsificado.

    Parámetros:
        ruta_referencia : ruta a la imagen de la firma de referencia
        ruta_sospechosa : ruta a la imagen de la firma a verificar
        modelo          : modelo Red Siamesa entrenado

    Retorna:
        dict con keys: 'resultado', 'score', 'alerta', 'nivel'
    """
    img_ref  = preprocesar_imagen(ruta_referencia)
    img_sosp = preprocesar_imagen(ruta_sospechosa)

    score = modelo.predict(
        [img_ref.reshape(1, 105, 105, 1),
         img_sosp.reshape(1, 105, 105, 1)],
        verbose=0
    )[0][0]

    if score < UMBRAL_GENUINO:
        resultado, alerta, nivel = 'GENUINO',     'Aprobación automática',       '✅'
    elif score <= UMBRAL_FALSIFICADA:
        resultado, alerta, nivel = 'INCIERTO',    'Derivar a revisión humana',   '⚠️'
    else:
        resultado, alerta, nivel = 'FALSIFICADO', 'Rechazo y alerta de fraude',  '🚨'

    return {'resultado': resultado, 'score': float(score),
            'alerta': alerta,       'nivel': nivel}


print('✅ Funciones definidas correctamente.')

## **3. Prueba de la función de predicción**
---

Verificamos que `verificar_firma()` funciona correctamente sobre pares del conjunto de test antes de desplegar la aplicación.

In [ ]:
import kagglehub

path      = kagglehub.dataset_download('robinreni/signature-verification-dataset')
test_path = os.path.join(path, 'sign_data/test')
test_csv  = pd.read_csv(os.path.join(path, 'sign_data/test_data.csv'), header=None)
test_csv.rename(columns={0: 'image_original_name',
                          1: 'image_comparison_name',
                          2: 'label'}, inplace=True)

print(f'Dataset cargado: {len(test_csv)} pares de test.')

In [ ]:
# Demostración sobre 6 pares del conjunto de test
print('=' * 65)
print('          DEMOSTRACIÓN: VERIFICACIÓN DE FIRMAS')
print('=' * 65)
print(f"  {'Par':>3} {'Real':<14} {'Score':>6}  {'Resultado':<14} {'Acción'}")
print('-' * 65)

for i, (_, row) in enumerate(test_csv.head(6).iterrows()):
    ruta_a = os.path.join(test_path, row['image_original_name'])
    ruta_b = os.path.join(test_path, row['image_comparison_name'])

    if not os.path.exists(ruta_a) or not os.path.exists(ruta_b):
        print(f'  {i:>3}  ⚠️  Imagen no encontrada — omitiendo par.')
        continue

    real = 'Genuino' if row['label'] == 0 else 'Falsificado'
    res  = verificar_firma(ruta_a, ruta_b, modelo)
    print(f"  {i:>3} {real:<14} {res['score']:>6.3f}  "
          f"{res['nivel']} {res['resultado']:<12} {res['alerta']}")

print('=' * 65)

## **4. Visualización de pares verificados**
---

Mostramos visualmente los pares procesados con su resultado para verificar el correcto funcionamiento del pipeline completo.

In [ ]:
# Visualizar 3 pares con su resultado
pares_validos = []
for _, row in test_csv.iterrows():
    ruta_a = os.path.join(test_path, row['image_original_name'])
    ruta_b = os.path.join(test_path, row['image_comparison_name'])
    if os.path.exists(ruta_a) and os.path.exists(ruta_b):
        pares_validos.append((ruta_a, ruta_b, row['label']))
    if len(pares_validos) == 3:
        break

fig, axes = plt.subplots(3, 2, figsize=(8, 10))
fig.suptitle('Demostración visual — Red Siamesa: Verificación de Firmas', fontsize=12)

for i, (ruta_a, ruta_b, label_real) in enumerate(pares_validos):
    img_a = cv2.imread(ruta_a, cv2.IMREAD_GRAYSCALE)
    img_b = cv2.imread(ruta_b, cv2.IMREAD_GRAYSCALE)
    res   = verificar_firma(ruta_a, ruta_b, modelo)
    real  = 'Genuino' if label_real == 0 else 'Falsificado'
    color = 'green' if res['resultado'] == 'GENUINO' else \
            'orange' if res['resultado'] == 'INCIERTO' else 'red'

    axes[i, 0].imshow(img_a, cmap='gray')
    axes[i, 0].set_title(f'Par {i+1} — Referencia (Real: {real})', fontsize=9)
    axes[i, 0].axis('off')

    axes[i, 1].imshow(img_b, cmap='gray')
    axes[i, 1].set_title(
        f"{res['nivel']} {res['resultado']}  |  Score: {res['score']:.3f}\n{res['alerta']}",
        fontsize=9, color=color
    )
    axes[i, 1].axis('off')

plt.tight_layout()
plt.show()

## **5. Generación de archivos para el despliegue**
---

Generamos los tres archivos necesarios para desplegar la app en Streamlit Cloud:
- `app.py` — aplicación Streamlit
- `requirements.txt` — dependencias del proyecto
- Verificación de que `modelo_siames_firmas_final.keras` está presente

In [ ]:
app_code = r'''
import streamlit as st
import keras
import numpy as np
import cv2
from PIL import Image
import os

st.set_page_config(
    page_title="Verificador de Firmas",
    page_icon="✍️",
    layout="centered"
)

MODEL_PATH         = os.getenv("MODEL_PATH",         "modelo_siames_firmas_final.keras")
UMBRAL_GENUINO     = float(os.getenv("UMBRAL_GENUINO",     0.40))
UMBRAL_FALSIFICADA = float(os.getenv("UMBRAL_FALSIFICADA", 0.60))

@st.cache_resource
def cargar_modelo():
    if not os.path.exists(MODEL_PATH):
        st.error(f"Modelo no encontrado en '{MODEL_PATH}'.")
        st.stop()
    return keras.models.load_model(MODEL_PATH)

def preprocesar_desde_pil(imagen_pil, tam=(105, 105)):
    img = np.array(imagen_pil.convert("L"))
    img = cv2.resize(img, tam)
    img = img / 255.0
    return img.reshape(tam[0], tam[1], 1).astype(np.float32)

def verificar_firma(img_ref, img_sosp, modelo):
    score = modelo.predict(
        [img_ref.reshape(1, 105, 105, 1), img_sosp.reshape(1, 105, 105, 1)],
        verbose=0
    )[0][0]
    if score < UMBRAL_GENUINO:
        return "✅ GENUINO",     float(score), "Aprobación automática",      "green"
    elif score <= UMBRAL_FALSIFICADA:
        return "⚠️ INCIERTO",   float(score), "Derivar a revisión humana",  "orange"
    else:
        return "🚨 FALSIFICADO", float(score), "Rechazo y alerta de fraude", "red"

st.title("✍️ Verificador de Firmas Adulteradas")
st.markdown(
    "Sistema de apoyo para la detección de firmas falsificadas basado en "
    "**Redes Siamesas con CNN** (AUC-ROC = 0.8184). "
    "Suba las dos imágenes de firma y obtenga el resultado de verificación."
)
st.divider()

col1, col2 = st.columns(2)
with col1:
    st.subheader("Firma de referencia")
    st.caption("Firma conocida como genuina (registro del cliente)")
    archivo_ref = st.file_uploader("Subir firma de referencia",
                                    type=["png","jpg","jpeg"], key="ref")
    if archivo_ref:
        st.image(Image.open(archivo_ref), caption="Firma de referencia",
                 use_column_width=True)

with col2:
    st.subheader("Firma sospechosa")
    st.caption("Firma extraída del documento a verificar")
    archivo_sosp = st.file_uploader("Subir firma sospechosa",
                                     type=["png","jpg","jpeg"], key="sosp")
    if archivo_sosp:
        st.image(Image.open(archivo_sosp), caption="Firma sospechosa",
                 use_column_width=True)

st.divider()

if st.button("🔍 Verificar firma", type="primary", use_container_width=True):
    if archivo_ref is None or archivo_sosp is None:
        st.error("Por favor suba ambas imágenes antes de verificar.")
    else:
        with st.spinner("Analizando el par de firmas..."):
            modelo_cargado = cargar_modelo()
            img_ref_arr  = preprocesar_desde_pil(Image.open(archivo_ref))
            img_sosp_arr = preprocesar_desde_pil(Image.open(archivo_sosp))
            resultado, score, alerta, color = verificar_firma(
                img_ref_arr, img_sosp_arr, modelo_cargado
            )
        st.markdown(f"### Resultado: :{color}[{resultado}]")
        col_a, col_b = st.columns(2)
        with col_a:
            st.metric("Score de falsificación", f"{score:.4f}")
        with col_b:
            st.metric("Umbrales", f"{UMBRAL_GENUINO} / {UMBRAL_FALSIFICADA}")
        st.info(f"**Acción recomendada:** {alerta}")
        with st.expander("ℹ️ Cómo interpretar el score"):
            st.markdown(
                f"| Score | Resultado | Acción |\n"
                f"|---|---|---|\n"
                f"| < {UMBRAL_GENUINO} | ✅ GENUINO | Aprobación automática |\n"
                f"| {UMBRAL_GENUINO}–{UMBRAL_FALSIFICADA} | ⚠️ INCIERTO | Revisión humana |\n"
                f"| > {UMBRAL_FALSIFICADA} | 🚨 FALSIFICADO | Rechazo + alerta |"
            )
        st.divider()
        st.caption(
            "⚠️ Este sistema es una herramienta de apoyo a la decisión, "
            "no reemplaza al perito calígrafo."
        )
'''

with open('app.py', 'w', encoding='utf-8') as f:
    f.write(app_code)

print('✅ app.py generado.')
print(f'   Líneas: {len(app_code.splitlines())}')

In [ ]:
# Generar requirements.txt
requirements = (
    'tensorflow>=2.13.0\n'
    'opencv-python-headless>=4.8.0\n'
    'numpy>=1.24.0\n'
    'streamlit>=1.32.0\n'
    'Pillow>=10.0.0\n'
    'kagglehub>=0.2.0\n'
    'pandas>=2.0.0\n'
    'scikit-learn>=1.3.0\n'
)

with open('requirements.txt', 'w') as f:
    f.write(requirements)

print('✅ requirements.txt generado.')
print(requirements)

In [ ]:
# Verificar que todos los archivos para el despliegue están presentes
archivos_requeridos = [
    ('app.py',                           'Aplicación Streamlit'),
    ('requirements.txt',                 'Dependencias del proyecto'),
    ('modelo_siames_firmas_final.keras', 'Modelo entrenado Red Siamesa'),
]

print('Verificación de archivos para el despliegue:')
print('-' * 55)
todos_ok = True
for archivo, descripcion in archivos_requeridos:
    existe = os.path.exists(archivo)
    estado = '✅' if existe else '❌ FALTANTE'
    tam    = f'({os.path.getsize(archivo)/1024:.1f} KB)' if existe else ''
    print(f'  {estado}  {archivo:<40} {tam}')
    if not existe:
        todos_ok = False

print()
if todos_ok:
    print('✅ Todos los archivos están listos para el despliegue.')
    print('   Siguiente paso: hacer push a GitHub y desplegar en Streamlit Cloud.')
else:
    print('⚠️  Algunos archivos faltan. Revisar antes de continuar.')

## **6. Despliegue en Streamlit Cloud**
---

Una vez verificados los archivos, seguir estos pasos para desplegar la app:

1. Descargar `app.py` y `requirements.txt` desde Colab (celda siguiente).
2. Agregar ambos archivos a la raíz del repositorio GitHub `iagp-mlds6-proyecto`.
3. Asegurarse de que `modelo_siames_firmas_final.keras` también está en la raíz del repositorio.
4. Ir a [share.streamlit.io](https://share.streamlit.io) → **New app**.
5. Seleccionar: repositorio `iagp-mlds6-proyecto`, rama `main`, archivo `app.py`.
6. Hacer clic en **Deploy**. La app estará disponible en `https://<nombre>.streamlit.app` en ~2 minutos.

> **Nota sobre el modelo:** GitHub tiene un límite de 100 MB por archivo. Si el modelo supera ese tamaño, usar [Git LFS](https://git-lfs.github.com/) o almacenarlo en Google Drive y cargarlo dinámicamente en `app.py`.

In [ ]:
# Descargar app.py, requirements.txt y el modelo desde Colab
from google.colab import files

for archivo in ['app.py', 'requirements.txt', 'modelo_siames_firmas_final.keras']:
    if os.path.exists(archivo):
        files.download(archivo)
        print(f'✅ Descargado: {archivo}')
    else:
        print(f'❌ No encontrado: {archivo}')

# **Créditos**
---

* **Profesor:** [Fabio Augusto Gonzalez](https://dis.unal.edu.co/~fgonza/)
* **Asistentes docentes:**
  * [Santiago Toledo Cortés](https://sites.google.com/unal.edu.co/santiagotoledo-cortes/)
* **Diseño de imágenes:**
    - [Mario Andres Rodriguez Triana](mailto:mrodrigueztr@unal.edu.co).
* **Coordinador de virtualización:**
    - [Edder Hernández Forero](https://www.linkedin.com/in/edder-hernandez-forero-28aa8b207/).

**Universidad Nacional de Colombia** - *Facultad de Ingeniería*